# Nigeria Agricultural Commodity Market Price Analysis

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook.

## Important data note
The included CSV is a **synthetic demonstration dataset** created for learning and portfolio practice. It is not an official survey and must not be described as actual observed Nigerian market prices. Replace it with verified market data before making real-world claims.

## Objectives
- Inspect and clean tabular market-price records.
- Compare average prices by commodity and market/state.
- Explore monthly price movement.
- Communicate patterns with charts.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'market_prices_illustrative.csv'
OUT = ROOT / 'visualizations'
OUT.mkdir(exist_ok=True)
df_raw = pd.read_csv(DATA)
df_raw.head()


## 1. Initial inspection
Check dataset dimensions, column types, missing values and duplicate records before analysis.


In [ ]:
print('Shape:', df_raw.shape)
display(df_raw.info())
display(df_raw.isna().sum().to_frame('missing_values'))
print('Duplicate rows:', df_raw.duplicated().sum())


## 2. Data cleaning
Convert dates and prices to appropriate types, remove duplicate rows, and exclude records without the fields required for price analysis.


In [ ]:
df = df_raw.copy()
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['price_ngn'] = pd.to_numeric(df['price_ngn'], errors='coerce')
df = df.drop_duplicates()
df = df.dropna(subset=['date','state','market','commodity','price_ngn'])
df = df[df['price_ngn'] > 0].copy()
df['month'] = df['date'].dt.to_period('M').astype(str)
print('Cleaned shape:', df.shape)
df.head()


## 3. Descriptive statistics and commodity comparison


In [ ]:
display(df['price_ngn'].describe().to_frame('price_ngn'))
commodity_summary = df.groupby('commodity')['price_ngn'].agg(['mean','median','min','max','count']).sort_values('mean', ascending=False)
display(commodity_summary.round(2))
commodity_summary.to_csv(OUT / 'commodity_summary.csv')


## 4. Visualisations


In [ ]:
plt.figure(figsize=(10,6))
df.groupby('commodity')['price_ngn'].mean().sort_values().plot(kind='barh')
plt.title('Average Illustrative Price by Commodity')
plt.xlabel('Average price (NGN)'); plt.ylabel('')
plt.tight_layout(); plt.show()


In [ ]:
monthly = df.groupby(['month','commodity'], as_index=False)['price_ngn'].mean()
plt.figure(figsize=(12,6))
monthly = df.groupby(['month','commodity'])['price_ngn'].mean().unstack(); monthly.plot(figsize=(12,6))
plt.title('Illustrative Monthly Commodity Price Trends')
plt.xlabel('Month'); plt.ylabel('Average price (NGN)')
plt.xticks(rotation=45); plt.tight_layout(); plt.show()


In [ ]:
state_avg = df.groupby('state', as_index=False)['price_ngn'].mean().sort_values('price_ngn', ascending=False)
plt.figure(figsize=(9,5))
state_avg.set_index('state')['price_ngn'].plot(kind='barh')
plt.title('Average Illustrative Prices by State')
plt.xlabel('Average price (NGN)'); plt.ylabel('')
plt.tight_layout(); plt.show()


In [ ]:
pivot = df.pivot_table(index='state', columns='commodity', values='price_ngn', aggfunc='mean')
plt.figure(figsize=(11,6))
plt.imshow(pivot.values, aspect='auto'); plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha='right'); plt.yticks(range(len(pivot.index)), pivot.index); plt.colorbar()
plt.title('Average Illustrative Commodity Prices by State')
plt.tight_layout(); plt.show()


## 5. Findings and limitations
Write your own findings after running the notebook. Discuss which commodities have the highest average demonstration prices, how the monthly series changes, and how state averages differ.

**Limitation:** Because the included records are synthetic, these charts demonstrate analytical methods only. They do not establish actual market prices, inflation, or differences between Nigerian markets. Use a documented, verified dataset for factual conclusions.
